## Middleware


In [1]:
import os 
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)



In [3]:
config ={"configurable":{"thread_id":"test_1"}}


In [4]:
# questions = [
#     "What is 2+2?",
#     "What is 10*5?",
#     "What is 100/4?",
#     "What is 15-7?",
#     "What is 3*3?",
#     "What is 4*4?",
# ]

# for q in questions:
#     response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
#     print(f"Messages : {response}")
#     print(f"Messages: {len(response["messages"])}")

## Token size

In [5]:
# from langchain.tools import tool

# @tool
# def searchHotel(city:str)->str:
#     """Search hotels - returns long response to use more tokens"""
#     return f"""Hotels in {city}:
#      1. Grand Hotel - 5 star, $350/night, spa, pool, gym
#     2. City Inn - 4 star, $180/night, business center
#     3. Budget Stay - 3 star, $75/night, free wifi"""

# agent1 = create_agent(
#     model = "groq:qwen/qwen3-32b",
#     tools = [searchHotel],
#     checkpointer = InMemorySaver(),
#     middleware = [
#         SummarizationMiddleware(
#         model = "groq:qwen/qwen3-32b",
#         trigger = ("tokens",550),
#         keep=("tokens",200)
#     )
#     ]
# )

# config1 = {"configurable":{"thread_id":"test_2"}}

# def counttoken(messages):
#     total_chars = sum(len(str(m.content))for m in messages)
#     return total_chars // 4


In [6]:
# cities = ["PAris","Karachi","Islamabad", "New York", "Dubai", "Singapore"]

# for city in cities:
#     response = agent1.invoke({"messages":[HumanMessage(content=f"Find hotels in {city}")]},config=config1)

#     tokens = counttoken(response["messages"])
#     print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
#     print(f"{(response['messages'])}")

In [7]:

# @tool
# def searchHotel(city:str)->str:
#     """Search hotels - returns long response to use more tokens"""
#     return f"""Hotels in {city}:
#      1. Grand Hotel - 5 star, $350/night, spa, pool, gym"""

# agent2 = create_agent(
#     model = "groq:qwen/qwen3-32b",
#     tools = [searchHotel],
#     checkpointer = InMemorySaver(),
#     middleware = [
#         SummarizationMiddleware(
#         model = "groq:qwen/qwen3-32b",
#         trigger = ("fraction",0.005),
#         keep=("fraction",0.002)
#     )
#     ]
# )

# config2 = {"configurable":{"thread_id":"test_2"}}

# def counttoken(messages):
#     total_chars = sum(len(str(m.content))for m in messages)
#     return total_chars // 4


# cities = ["Paris","London","Tokyo", "New York", "Dubai", "Singapore"]

# for city in cities:
#     response = agent2.invoke({"messages":[HumanMessage(content=f"Find hotels in {city}")]},config=config2)

#     tokens = counttoken(response["messages"])
#     fraction = tokens/128000
#     print(f"{city}: ~{tokens} tokens, ({fraction:.4%}),{len(response["messages"])} msgs")
#     print(f"{(response['messages'])}")

## Human in the loop middleware

In [8]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email(email_id:str)->str:
    """Mock function to read an email by its id"""
    return f"Email content for id : {email_id}"

def send_email(recipent:str,subject:str,body:str)->str:
    """Mock function to send an email """
    return f"Email sent to {recipent} with subject {subject}"
    

In [9]:
agent4 = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [read_email,send_email],
    checkpointer = InMemorySaver(),
    middleware= [HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email":{
                "allowed_decisions":["approve","edit","reject"]
            },
            "read_email" : False
        }
    )
    ]
)

In [10]:
config4 = {"configurable":{"thread_id":"test_4"}}

result = agent4.invoke(
    {"messages":[HumanMessage(content="Send an email to john@test.com with subject 'hello' and body 'how are you'")]},
    config = config4
)

In [11]:
result

{'messages': [HumanMessage(content="Send an email to john@test.com with subject 'hello' and body 'how are you'", additional_kwargs={}, response_metadata={}, id='bb157251-81bb-4edc-8154-1d41f05f0b56'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'hello' and body 'how are you'. Let me check the available tools. There's the send_email function. It requires recipent, subject, and body. All three are provided here. The parameters need to be in a JSON object. So I'll call send_email with those arguments. No need for the read_email function here since the user isn't asking to read an email. Just need to structure the tool_call correctly with the provided parameters.\n", 'tool_calls': [{'id': 'dz1tn0525', 'function': {'arguments': '{"body":"how are you","recipent":"john@test.com","subject":"hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 1

In [12]:
# from langgraph.types import Command

# if '__interrupt__' in result:
#     print("Paused! Approving...")

#     result = agent4.invoke(
#         Command(
#             resume={
#                 "decisions":[
#                     {"type":"approve"}
#                 ]
#             }
#         ),
#         config = config4
#     )

#     print(f"Result : {result['messages'][-1].content}")

In [13]:
# result

In [14]:
# from langgraph.types import Command

# if '__interrupt__' in result:
#     print("Paused! Rejecting...")

#     result = agent4.invoke(
#         Command(
#             resume={
#                 "decisions":[
#                     {"type":"reject"}
#                 ]
#             }
#         ),
#         config = config4
#     )

#     print(f"Result : {result['messages'][-1].content}")

In [15]:
# result

In [16]:
from langgraph.types import Command

if '__interrupt__' in result:
    print("Paused! Editing...")

    result = agent4.invoke(
        Command(
            resume={
                "decisions":[{
                    "type":"edit",
                    "edited_action":{
                        "name":"send_email",
                        "args": {
                            "recipent":"correct@test.com",
                            "subject":"Corrected Subject",
                            "body":"This was edited before sending"
                        }
                    }
                    }
                ]
            }
        ),
        config = config4
    )

    print(f"Result : {result['messages'][-1].content}")

Paused! Editing...
Result : The email has been successfully sent to **correct@test.com** with the subject **"Corrected Subject"** and the body **"This was edited before sending"**. Let me know if you need any further assistance!


In [17]:
result

{'messages': [HumanMessage(content="Send an email to john@test.com with subject 'hello' and body 'how are you'", additional_kwargs={}, response_metadata={}, id='bb157251-81bb-4edc-8154-1d41f05f0b56'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'hello' and body 'how are you'. Let me check the available tools. There's the send_email function. It requires recipent, subject, and body. All three are provided here. The parameters need to be in a JSON object. So I'll call send_email with those arguments. No need for the read_email function here since the user isn't asking to read an email. Just need to structure the tool_call correctly with the provided parameters.\n", 'tool_calls': [{'id': 'dz1tn0525', 'function': {'arguments': '{"body":"how are you","recipent":"john@test.com","subject":"hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 1